# Quantum Sentiment Analysis - walkthrough

This notebook only *demonstrates* the components of the `qsa` package; all the logic lives in the Python modules.
Pipeline: **sentence → GloVe-SIF embedding → PCA + Fisher ranking to 2ⁿ dimensions (here n = 1: two numbers per sentence) → amplitude encoding (Grover–Rudolph) → interference classifier**.

Prerequisite: `python scripts/download_glove.py --dim 300` (the embeddings are cached after the first run).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # run from notebooks/
import os; os.chdir(pathlib.Path.cwd().parent)
import numpy as np
from qsa.data import load_sst2
from qsa.embeddings import EmbeddingCache, GloVeSIFEmbedder
from qsa.experiment import build_representation
from qsa.quantum import InterferenceClassifier, GroverRudolphLoader, pair_circuit
from qsa.baselines import LogisticProbe, NearestCentroid
from qsa.metrics import classification_metrics

## 1. Data and sentence embeddings

In [2]:
data = load_sst2("data/sst2")
display(data.describe())
emb = EmbeddingCache().get_or_compute(GloVeSIFEmbedder(), data)
{split: x.shape for split, x in emb.items()}

,split,sentences,positive,negative,mean_tokens
0,train,6920,3610,3310,18.51
1,dev,872,444,428,18.73
2,test,1821,909,912,18.48


{'train': (6920, 300), 'test': (1821, 300), 'dev': (872, 300)}

## 2. Compression to 2ⁿ dimensions and amplitude encoding
PCA and the Fisher ranking are fitted on the training split only; the normaliser maps every vector to a unit vector, i.e. a valid real quantum state of n qubits.

In [3]:
n = 1   # one data qubit, as in the report
rep = build_representation("glove-sif", emb, data.train.labels, "pca-fisher", n)
print(f"k = {rep.k} dimensions, variance kept = {rep.variance_retention:.3f}")
print("norms:", np.linalg.norm(rep.states["test"][:5], axis=1))

k = 2 dimensions, variance kept = 0.061
norms: [1. 1. 1. 1. 1.]


## 3. Grover–Rudolph state preparation
The loader prepares the state exactly (fidelity 1) with 2ⁿ−1 RY rotations and at most 2ⁿ−2 CNOTs.

In [4]:
loader = GroverRudolphLoader()
state = rep.states["test"][0]
qc = loader.circuit(state)
print("fidelity:", round(loader.fidelity(qc, state), 12), "| ops:", dict(qc.count_ops()))
qc.draw(output="text")

fidelity: 1.0 | ops: {'ry': 1}


┌────────────┐
q: ┤ Ry(1.5129) ├
   └────────────┘

## 4. The interference circuit for one (negative, positive) pair

In [5]:
x0 = rep.states["train"][data.train.labels == 0][0]
x1 = rep.states["train"][data.train.labels == 1][0]
pc = pair_circuit(state, x0, x1)
print(pc.num_qubits, "qubits")
pc.draw(output="text", fold=120)

4 qubits


┌───┐┌───┐      ┌───┐                 ┌───┐┌─┐   
    a: ┤ H ├┤ X ├──■───┤ X ├──■──────────■───┤ H ├┤M├───
       ├───┤├───┤  │   └───┘  │   ┌───┐  │   └───┘└╥┘   
    m: ┤ H ├┤ X ├──┼──────────■───┤ X ├──■─────■───╫────
       └───┘└───┘┌─┴──┐     ┌─┴──┐└───┘┌─┴──┐  │   ║    
    g: ──────────┤ Ut ├─────┤ U0 ├─────┤ U1 ├──┼───╫────
                 └────┘     └────┘     └────┘┌─┴─┐ ║ ┌─┐
    c: ──────────────────────────────────────┤ X ├─╫─┤M├
                                             └───┘ ║ └╥┘
out: 2/════════════════════════════════════════════╩══╩═
                                                   0  1

## 5. Closed form vs. Qiskit Aer
`P(a=0, c) = (1 + <t|x_c>)/4`: the fast backend samples shots from these probabilities; here we check it against the circuit simulation on a few sentences.

In [6]:
train, ytr = rep.states["train"], data.train.labels
test, yte = rep.states["test"][:20], data.test.labels[:20]
kw = dict(protocol="pairs", n_pairs=5, shots=4096, seed=0)
aer = InterferenceClassifier(backend="aer", **kw).fit(train, ytr)
counts = aer._aer_counts(test)
exact = aer.exact_probabilities(test)
print("max |frequency - probability| =", np.abs(counts / 4096 - exact).max().round(4))

max |frequency - probability| = 0.0236


## 6. Full test set: quantum classifier vs. classical counterparts

In [7]:
rows = {}
for name, clf in [("centroid pair, 1024 shots", InterferenceClassifier("centroid", shots=1024)),
                  ("centroid pair, 16 shots", InterferenceClassifier("centroid", shots=16)),
                  ("pairs, 1024 shots", InterferenceClassifier("pairs", n_pairs=100, shots=1024)),
                  ("superposition, exact", InterferenceClassifier("superposition", shots=None))]:
    rows[name] = classification_metrics(data.test.labels, clf.fit(train, ytr).predict(rep.states["test"]))
rows["nearest centroid"] = classification_metrics(data.test.labels, NearestCentroid().fit(train, ytr).predict(rep.states["test"]))
probe = LogisticProbe().fit(rep.compressed["train"], ytr, rep.compressed["dev"], data.dev.labels)
rows["logistic regression"] = classification_metrics(data.test.labels, probe.predict(rep.compressed["test"]))
import pandas as pd
pd.DataFrame(rows).T.round(3)

,accuracy,precision,recall,f1,mcc
"centroid pair, 1024 shots",0.702,0.701,0.703,0.702,0.405
"centroid pair, 16 shots",0.686,0.684,0.692,0.688,0.373
"pairs, 1024 shots",0.701,0.702,0.695,0.699,0.401
"superposition, exact",0.700,0.698,0.701,0.700,0.399
nearest centroid,0.700,0.698,0.701,0.700,0.399
logistic regression,0.708,0.690,0.754,0.720,0.418


With one data qubit every rule agrees within about one point: two normalised numbers are an angle, and any linear rule only picks an arc of the circle. The exact superposition protocol coincides with the nearest centroid (in the infinite-shot limit the circuit *is* a nearest-centroid classifier), and the centroid pair is already close to its limit with 16 shots.